# Stablecoin StressBench — Evaluation & Leaderboard

This notebook evaluates all trained models and produces the benchmark leaderboard:
- ML metrics: AUROC, AUPRC, F1, Brier score
- Economic metrics: net bps captured, hit rate, Sharpe ratio, max drawdown
- Leaderboard ranking

In [ ]:
import numpy as np
from stressbench.evaluation.backtest import run_backtest
from stressbench.evaluation.leaderboard import build_leaderboard, print_leaderboard
from stressbench.models.baselines import LogisticBaseline, LastValueBaseline
from stressbench.models.tree_models import RandomForestWrapper

rng = np.random.default_rng(99)
n_test = 1000
X_test = rng.standard_normal((n_test, 20)).astype(np.float32)
y_test = (rng.standard_normal(n_test) > 0).astype(np.int8)
y_net = rng.normal(5, 15, n_test)  # Net profit in bps

# Train and evaluate
X_train = rng.standard_normal((3000, 20)).astype(np.float32)
y_train = (rng.standard_normal(3000) > 0).astype(np.int8)

results = []
for name, model in [
    ('last_value', LastValueBaseline()),
    ('logistic', LogisticBaseline()),
    ('rf', RandomForestWrapper(task='classification', n_estimators=50)),
]:
    model.fit(X_train, y_train)
    result = run_backtest(model, X_test, y_test, y_net, model_name=name)
    results.append(result)

leaderboard = build_leaderboard(results)
print_leaderboard(leaderboard)

In [ ]:
import matplotlib.pyplot as plt
from stressbench.evaluation.economic_metrics import cumulative_pnl

fig, ax = plt.subplots(figsize=(12, 5))
for r in results:
    # Reconstruct signal from model
    model_name = r['model']
    econ = r['economic_metrics']
    ax.set_title('Cumulative P&L by Model')
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Cumulative P&L (USD)')

ax.legend()
plt.tight_layout()
plt.show()